### XTTS modified model for generating Ombré samples
##### use with modified XTTS.py stored in the model folder

In [ ]:
import IPython.display as ipd
import numpy as np
import os
import pandas as pd

In [ ]:
#based on the published XTTS implementation https://github.com/coqui-ai/TTS
from TTS.utils.manage import ModelManager
from TTS.config import load_config
from TTS.tts.models import setup_model as setup_tts_model

In [ ]:
manager = ModelManager(models_file="TTS/.models.json", progress_bar=True, verbose=False)
model_name = "tts_models/multilingual/multi-dataset/xtts_v2"

In [ ]:
def download_model_by_name(model_name: str):
    model_path, config_path, model_item = manager.download_model(model_name)
    if "fairseq" in model_name or (model_item is not None and isinstance(model_item["model_url"], list)):
        # return model directory if there are multiple files
        # we assume that the model knows how to load itself
        return None, None, None, None, model_path
    if model_item.get("default_vocoder") is None:
        return model_path, config_path, None, None, None
    vocoder_path, vocoder_config_path, _ = manager.download_model(model_item["default_vocoder"])
    return model_path, config_path, vocoder_path, vocoder_config_path, None

In [ ]:
model_path, config_path, vocoder_path, vocoder_config_path, model_dir = download_model_by_name(model_name)

In [ ]:
#load fine tuned model on VCTK corpus, can be skipped for using the published model
model_dir = "run/training/GPT_XTTS_v2.0_finetuned_model_location"

In [ ]:
config = load_config(os.path.join(model_dir, "config.json"))
tts_model = setup_tts_model(config)

In [ ]:
# optional if vocab.json is missing in the folder of the fine-tuned model, copy it there
#import shutil
#shutil.copyfile(os.path.join("run/training/XTTS_v2.0_original_model_files", "vocab.json"),os.path.join(model_dir, "vocab.json"))

In [ ]:
tts_model.load_checkpoint(config, checkpoint_path=model_dir+"/best_model.pth",
                         checkpoint_dir=model_dir, eval=True)
#tts_model.load_checkpoint(config, checkpoint_dir=model_dir, eval=True)

In [ ]:
# optional: load the model to gpu for faster inference
tts_model.cuda()

### Ombré synthesis

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# function to time align latents and transition from one to the other over time
def blend_latents_numpy(a_np, b_np):
    # Convert from numpy to torch tensors
    a = torch.from_numpy(a_np).float().unsqueeze(0)  # [1, T_a, 1024]
    b = torch.from_numpy(b_np).float().unsqueeze(0)  # [1, T_b, 1024]

    T_a = a.shape[1]
    T_b = b.shape[1]
    T_m = int((T_a + T_b) / 2)

    # Interpolate to T_m
    a_interp = F.interpolate(a.transpose(1, 2), size=T_m, mode='linear', align_corners=True).transpose(1, 2)
    b_interp = F.interpolate(b.transpose(1, 2), size=T_m, mode='linear', align_corners=True).transpose(1, 2)

    # Create weights from 0 to 1 across T_m and blend
    weights = torch.linspace(0, 1, T_m).view(1, T_m, 1)
    blended = (1 - weights) * a_interp + weights * b_interp  # [1, T_m, 1024]

    return blended

In [ ]:
#create references (for Ombré) and an equal weighted output (for vocoder input)
refwav = ["./ref/vctk/p226_023.wav", "./ref/vctk/p262_023.wav"]
wghts = [[1.7, 1.0], [1.0, 1.7], [0.5, 0.5]]
audios = []
for i in range(len(wghts)):
    audio = tts_model.synthesize(
        text="Implicit bias is like the invisible weight of water, pushing down on us all the time, but only noticeable when we try to swim upstream.",
        config=config,
        speaker_id=None,
        voice_dirs=None,
        d_vector=None,
        temperature=0.9,
        #speed=1.1,
        repetition_penalty=20.0,
        speaker_wav=refwav,
        audio_weights=wghts[i],
        language="en")
    audio_out = np.asarray(audio['wav'])
    print(f'weight: {wghts[i]}:')
    display(ipd.Audio(audio_out, rate=24000, autoplay=False))
    audios.append(audio)

In [ ]:
ombre_latent = blend_latents_numpy(audios[0]['gpt_latents'][0], audios[1]['gpt_latents'][0])
inverse_latent = blend_latents_numpy(audios[1]['gpt_latents'][0], audios[0]['gpt_latents'][0])

In [ ]:
# create ombré audio
with torch.no_grad():
    wav_ombre = tts_model.hifigan_decoder(ombre_latent, g=audios[2]['speaker_embedding']).cpu().squeeze()
    display(ipd.Audio(wav_ombre, rate=24000, autoplay=False))
    inverse_ombre = tts_model.hifigan_decoder(inverse_latent, g=audios[2]['speaker_embedding']).cpu().squeeze()
    display(ipd.Audio(inverse_ombre, rate=24000, autoplay=False))
    